# Benchamark utilizando AutoML. En este caso con flaml.

In [1]:
# Cargar librerías necesarias

!pip install -U "flaml[automl]"
import numpy as np
import pandas as pd
if not hasattr(np, "NaN"): np.NaN = np.nan  # hotfix NumPy 2.x
from flaml import AutoML
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score, f1_score, precision_score, recall_score,
    roc_auc_score, accuracy_score
)



In [2]:
uploaded = files.upload()

Saving df_trans.pkl to df_trans (2).pkl


In [3]:
df = pd.read_pickle("df_trans.pkl")

In [4]:
target = "Diabetes_binary"
X = df.drop(columns=[target])
y = df[target].astype(int)

# Split (estratificado recomendado en clasificación)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Configura y ejecuta FLAML (solo estimadores sklearn para evitar deps extra)
automl = AutoML()
settings = {
    "time_budget": 600,
    "task": "classification",
    "metric": "ap",          # PR-AUC
    "eval_method": "cv",
    "n_splits": 5,
    "seed": 42,
    "log_file_name": "flaml_automl.log",
    # Estimadores de sklearn (sin xgboost/lgbm/catboost):
    "estimator_list": ["lrl1","lrl2","rf","extra_tree","kneighbor","svc","sgd","histgb"],
}
automl.fit(X_train=X_train, y_train=y_train, **settings)

# Evaluación en test (métricas alineadas a tu benchmark)
y_pred  = automl.predict(X_test)
y_score = automl.predict_proba(X_test)[:, 1]

print("=== Métricas en TEST ===")
print("PR-AUC   :", average_precision_score(y_test, y_score))
print("F1       :", f1_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("ROC-AUC  :", roc_auc_score(y_test, y_score))
print("Accuracy :", accuracy_score(y_test, y_pred))

print("\n=== Modelo Seleccionado por FLAML ===")
print("Algoritmo:", automl.best_estimator)
print("Mejores hiperparámetros:\n", automl.best_config)

[flaml.automl.logger: 08-28 09:09:23] {1752} INFO - task = classification
[flaml.automl.logger: 08-28 09:09:23] {1763} INFO - Evaluation method: cv
[flaml.automl.logger: 08-28 09:09:23] {1862} INFO - Minimizing error metric: 1-ap
[flaml.automl.logger: 08-28 09:09:23] {1979} INFO - List of ML learners in AutoML Run: ['lrl1', 'lrl2', 'rf', 'extra_tree', 'kneighbor', 'svc', 'sgd', 'histgb']
[flaml.automl.logger: 08-28 09:09:23] {2282} INFO - iteration 0, current learner lrl1


INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 08-28 09:09:25] {2417} INFO - Estimated sufficient time budget=21659s. Estimated necessary time budget=22s.
[flaml.automl.logger: 08-28 09:09:25] {2466} INFO -  at 2.3s,	estimator lrl1's best error=0.2049,	best estimator lrl1's best error=0.2049
[flaml.automl.logger: 08-28 09:09:25] {2282} INFO - iteration 1, current learner svc


INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 08-28 09:09:26] {2466} INFO -  at 2.9s,	estimator svc's best error=0.2050,	best estimator lrl1's best error=0.2049
[flaml.automl.logger: 08-28 09:09:26] {2282} INFO - iteration 2, current learner sgd


INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 08-28 09:09:33] {2466} INFO -  at 10.1s,	estimator sgd's best error=0.2382,	best estimator lrl1's best error=0.2049
[flaml.automl.logger: 08-28 09:09:33] {2282} INFO - iteration 3, current learner histgb
[flaml.automl.logger: 08-28 09:09:34] {2466} INFO -  at 10.5s,	estimator histgb's best error=0.2978,	best estimator lrl1's best error=0.2049
[flaml.automl.logger: 08-28 09:09:34] {2282} INFO - iteration 4, current learner extra_tree
[flaml.automl.logger: 08-28 09:09:34] {2466} INFO -  at 10.9s,	estimator extra_tree's best error=0.2558,	best estimator lrl1's best error=0.2049
[flaml.automl.logger: 08-28 09:09:34] {2282} INFO - iteration 5, current learner rf
[flaml.automl.logger: 08-28 09:09:34] {2466} INFO -  at 11.2s,	estimator rf's best error=0.2523,	best estimator lrl1's best error=0.2049
[flaml.automl.logger: 08-28 09:09:34] {2282} INFO - iteration 6, current learner extra_tree
[flaml.automl.logger: 08-28 09:09:35] {2466} INFO -  at 11.5s,	estimator extra_tree

INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 08-28 09:09:49] {2466} INFO -  at 26.4s,	estimator lrl2's best error=0.2050,	best estimator lrl1's best error=0.2049
[flaml.automl.logger: 08-28 09:09:49] {2282} INFO - iteration 17, current learner kneighbor
[flaml.automl.logger: 08-28 09:09:52] {2466} INFO -  at 28.8s,	estimator kneighbor's best error=0.2937,	best estimator lrl1's best error=0.2049
[flaml.automl.logger: 08-28 09:09:52] {2282} INFO - iteration 18, current learner kneighbor
[flaml.automl.logger: 08-28 09:09:54] {2466} INFO -  at 31.3s,	estimator kneighbor's best error=0.2780,	best estimator lrl1's best error=0.2049
[flaml.automl.logger: 08-28 09:09:54] {2282} INFO - iteration 19, current learner histgb
[flaml.automl.logger: 08-28 09:09:55] {2466} INFO -  at 31.6s,	estimator histgb's best error=0.2189,	best estimator lrl1's best error=0.2049
[flaml.automl.logger: 08-28 09:09:55] {2282} INFO - iteration 20, current learner rf
[flaml.automl.logger: 08-28 09:09:55] {2466} INFO -  at 32.0s,	estimator r